In [1]:
STUDENT_ID = "TUPM-26-1227"     
SEED_DIGIT = int(STUDENT_ID[-1])

ID_SUM = sum(int(c) for c in STUDENT_ID if c.isdigit())

RECORDS_TO_PROCESS = 1000 + ID_SUM                 # first 1000 + ID_SUM data records
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2          # 2, 3, 4 or 5
TARGET_CLASS_LABEL = f"{TARGET_CLASS_DIGIT}xx"

print("STUDENT_ID          :", STUDENT_ID)
print("SEED_DIGIT          :", SEED_DIGIT)
print("ID_SUM              :", ID_SUM)
print("Records to process  :", RECORDS_TO_PROCESS)
print("Target status class :", TARGET_CLASS_LABEL)

STUDENT_ID          : TUPM-26-1227
SEED_DIGIT          : 7
ID_SUM              : 20
Records to process  : 1020
Target status class : 5xx


In [2]:
import os, csv, json, shutil
from collections import Counter

DATASET_PATH = "weblog.csv"

INPUT_DIR  = "data"
OUTPUT_DIR = "output"
BACKUP_DIR = "backup"
for d in (INPUT_DIR, OUTPUT_DIR, BACKUP_DIR):
    os.makedirs(d, exist_ok=True)

FILTERED_CSV = os.path.join(OUTPUT_DIR, f"filtered_{TARGET_CLASS_LABEL}.csv")
SUMMARY_JSON = os.path.join(OUTPUT_DIR, "summary.json")
BACKUP_JSON  = os.path.join(BACKUP_DIR, "summary_backup.json")

In [ ]:
DATASET_PATH = r"C:\Users\n0n3\Lab6_File_Handling_Solis\Lab6_File_Handling_Additional_Exercises\data\weblog.csv"

def find_col(header, *keys):
    for i, name in enumerate(header):
        n = name.strip().lower()
        if any(n.startswith(k) for k in keys):
            return i
    raise ValueError(f"Column {keys} not found in header: {header}")

records_processed = 0
records_in_class = 0
status_counts = Counter()
url_counts = Counter()
skipped_bad_rows = 0

try:
    with open(DATASET_PATH, "r", newline="", encoding="utf-8-sig") as src, \
        open(FILTERED_CSV, "w", newline="", encoding="utf-8") as out:

        reader = csv.reader(src)               
        writer = csv.writer(out)
        writer.writerow(["IP", "Time", "URL", "Status"])

        header = next(reader)
        i_ip   = find_col(header, "ip")
        i_time = find_col(header, "time")
        i_url  = find_col(header, "url")
        i_stat = find_col(header, "stat", "stau")       

        for row in reader:
            if records_processed >= RECORDS_TO_PROCESS:
                break
            records_processed += 1

            try:
                ip, ts, url, status = (row[i_ip].strip(), row[i_time].strip(),
                                    row[i_url].strip(), row[i_stat].strip())
                code_int = int(status)
            except (IndexError, ValueError):
                skipped_bad_rows += 1            # malformed / missing fields
                continue

            if code_int // 100 == TARGET_CLASS_DIGIT:
                writer.writerow([ip, ts, url, code_int])
                records_in_class += 1
                status_counts[code_int] += 1
                url_counts[url] += 1

top5_urls = url_counts.most_common(5)
print("Done. Rows read:", records_processed, "| malformed skipped:", skipped_bad_rows)

Done. Rows read: 1020 | malformed skipped: 0


In [4]:
summary = {
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "target_status_class": TARGET_CLASS_LABEL,
    "records_in_target_class": records_in_class,
    "count_per_status_code": {str(k): v for k, v in sorted(status_counts.items())},
    "five_most_requested_urls": [{"url": u, "count": c} for u, c in top5_urls],
    "filtered_csv_file": FILTERED_CSV,
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)
print("Saved:", SUMMARY_JSON)

Saved: output\summary.json


In [5]:
try:
    shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
except FileNotFoundError:
    print("Summary JSON is missing - nothing to back up.")
except PermissionError:
    print("Permission denied while creating the backup.")

backup_ok = os.path.isfile(BACKUP_JSON) and os.path.getsize(BACKUP_JSON) == os.path.getsize(SUMMARY_JSON)
print("Backup verified:", backup_ok)

Backup verified: True


In [6]:
counts_line = ", ".join(f"{k}: {v}" for k, v in sorted(status_counts.items())) or "none"
urls_line = "; ".join(f"{u} ({c})" for u, c in top5_urls) or "none"

print(f"Records processed     : {records_processed}")
print(f"Target status class   : {TARGET_CLASS_LABEL}")
print(f"Records in target class: {records_in_class}")
print(f"Count per status code : {counts_line}")
print(f"Five most requested URLs: {urls_line}")
print(f"Filtered CSV file     : {FILTERED_CSV}")
print(f"JSON summary file     : {SUMMARY_JSON}")
print(f"Backup verification   : {'Yes - ' + BACKUP_JSON + ' exists' if backup_ok else 'FAILED'}")

Records processed     : 1020
Target status class   : 5xx
Records in target class: 0
Count per status code : none
Five most requested URLs: none
Filtered CSV file     : output\filtered_5xx.csv
JSON summary file     : output\summary.json
Backup verification   : Yes - backup\summary_backup.json exists
